# 12 — Sliding Window + Hash Map

## Learning goals
- Understand fixed-size and variable-size sliding windows.
- Maintain character or value frequencies as a window moves.
- Solve substring and subarray problems using a hash map.
- Recognize when a window can shrink safely.
- Handle duplicates, negative values, and window boundaries correctly.

**Core idea:** A sliding window maintains information about a contiguous range. A hash map helps track frequencies or last-seen positions inside that range.

## 1. What is a sliding window?

A window is a contiguous part of an array or string.

Instead of recalculating information for every possible range, we move the window and update only what changed:
- Add the new rightmost element.
- Remove the element that leaves the left side.
- Update the answer.

This often reduces a brute-force `O(n²)` scan to `O(n)`.

## 2. Fixed-size window

The window always has exactly `k` elements.

Typical steps:
1. Build the first window.
2. Record its result.
3. Move right by one: add the new element and remove the outgoing element.
4. Update the result.

The window size does not change.

In [21]:
def max_sum_fixed_window(nums: list[int], k: int) -> int:
    if not 1 <= k <= len(nums):
        raise ValueError("k must be between 1 and len(nums)")

    window_sum = sum(nums[:k])
    best = window_sum

    for right in range(k, len(nums)):
        window_sum += nums[right] - nums[right - k]
        best = max(best, window_sum)

    return best

print(max_sum_fixed_window([2, 1, 5, 1, 3, 2], 3))  # 9

9


## 3. Variable-size window

The left and right boundaries move according to a condition.

A common pattern for **non-negative values** or many substring constraints:
1. Expand the right boundary.
2. Update the window state.
3. While the window violates a condition, move the left boundary and remove its contribution.
4. Update the answer when the window is valid.

Do not assume this works for every sum problem. With negative numbers, shrinking may increase a sum, so the usual sum-based window logic can fail.

## 4. Hash map inside a window

A frequency map tracks how many times each value currently appears.

When adding a value:
`freq[x] = freq.get(x, 0) + 1`

When removing a value:
- Decrement its count.
- Delete the key if the count becomes zero.

Deleting zero-count keys keeps the map representative of the current window.

In [22]:
def add(freq: dict, value) -> None:
    freq[value] = freq.get(value, 0) + 1

def remove(freq: dict, value) -> None:
    freq[value] -= 1
    if freq[value] == 0:
        del freq[value]

freq = {}
add(freq, "a")
add(freq, "b")
add(freq, "a")
print(freq)  # {'a': 2, 'b': 1}
remove(freq, "a")
print(freq)  # {'a': 1, 'b': 1}

{'a': 2, 'b': 1}
{'a': 1, 'b': 1}


## 5. Longest substring without repeating characters

Maintain a window containing no duplicate characters.

For each right character:
- If it was last seen inside the current window, move `left` past its previous position.
- Update its last-seen index.
- Update the maximum window length.

The last-seen dictionary stores the most recent index of each character.

In [23]:
def length_of_longest_substring(s: str) -> int:
    last_seen = {}
    left = 0
    best = 0

    for right, ch in enumerate(s):
        if ch in last_seen and last_seen[ch] >= left:
            left = last_seen[ch] + 1

        last_seen[ch] = right
        best = max(best, right - left + 1)

    return best

print(length_of_longest_substring("abcabcbb"))  # 3
print(length_of_longest_substring("bbbbb"))     # 1
print(length_of_longest_substring("pwwkew"))    # 3

3
1
3


## 6. Dry run — `s = "abba"`

Start with `left = 0`.

| right | char | Previous index | left after update | Current window | best |
|---:|---|---:|---:|---|---:|
| 0 | a | — | 0 | a | 1 |
| 1 | b | — | 0 | ab | 2 |
| 2 | b | 1 | 2 | b | 2 |
| 3 | a | 0 (outside window) | 2 | ba | 2 |

The important condition is `last_seen[ch] >= left`. A previous occurrence outside the current window must not move `left` backwards.

## 7. Longest substring with at most K distinct characters

Maintain a frequency map for the current window.

- Add the right character.
- While the number of distinct keys exceeds `k`, remove characters from the left.
- Record the longest valid window.

This is a classic variable-size window + frequency map pattern.

In [24]:
def longest_substring_at_most_k_distinct(s: str, k: int) -> int:
    if k <= 0:
        return 0

    freq = {}
    left = 0
    best = 0

    for right, ch in enumerate(s):
        freq[ch] = freq.get(ch, 0) + 1

        while len(freq) > k:
            outgoing = s[left]
            freq[outgoing] -= 1
            if freq[outgoing] == 0:
                del freq[outgoing]
            left += 1

        best = max(best, right - left + 1)

    return best

print(longest_substring_at_most_k_distinct("eceba", 2))  # 3
print(longest_substring_at_most_k_distinct("aa", 1))     # 2

3
2


## 8. Count substrings with at most K distinct characters

The key observation: if the window ending at `right` is valid and `left` is the smallest valid left boundary, then every start index from `left` through `right` also gives a valid substring ending at `right`.

So add `right - left + 1` to the answer at each step.

In [25]:
def count_substrings_at_most_k_distinct(s: str, k: int) -> int:
    if k < 0:
        return 0

    freq = {}
    left = 0
    answer = 0

    for right, ch in enumerate(s):
        freq[ch] = freq.get(ch, 0) + 1

        while len(freq) > k:
            outgoing = s[left]
            freq[outgoing] -= 1
            if freq[outgoing] == 0:
                del freq[outgoing]
            left += 1

        answer += right - left + 1

    return answer

print(count_substrings_at_most_k_distinct("pqpqs", 2))  # 12

12


## 9. Exactly K distinct characters

A useful identity:

`exactly(K) = at_most(K) - at_most(K - 1)`

Why? Every substring with at most `K` distinct characters belongs to either:
- at most `K - 1`, or
- exactly `K`.

Subtracting the first group leaves exactly `K`.

In [26]:
def count_substrings_exactly_k_distinct(s: str, k: int) -> int:
    if k <= 0:
        return 0

    return (
        count_substrings_at_most_k_distinct(s, k)
        - count_substrings_at_most_k_distinct(s, k - 1)
    )

print(count_substrings_exactly_k_distinct("pqpqs", 2))  # 7

7


## 10. Find all anagrams in a string

Given `s` and `p`, return all start indices where a substring of `s` is an anagram of `p`.

The window size is fixed at `len(p)`. Maintain character frequencies for the pattern and current window.

For lowercase English letters, two arrays of length 26 are simple and efficient. This assumption must be guaranteed by the problem.

In [27]:
def find_anagrams(s: str, p: str) -> list[int]:
    if len(p) > len(s):
        return []

    need = [0] * 26
    window = [0] * 26

    for ch in p:
        need[ord(ch) - 97] += 1

    k = len(p)
    answer = []

    for right, ch in enumerate(s):
        window[ord(ch) - 97] += 1

        if right >= k:
            outgoing = s[right - k]
            window[ord(outgoing) - 97] -= 1

        if right >= k - 1 and window == need:
            answer.append(right - k + 1)

    return answer

print(find_anagrams("cbaebabacd", "abc"))  # [0, 6]
print(find_anagrams("abab", "ab"))         # [0, 1, 2]

[0, 6]
[0, 1, 2]


## 11. Permutation in String

Return `True` if any substring of `s2` is a permutation of `s1`.

This is the same fixed-size frequency-window pattern as finding all anagram indices. Stop as soon as a matching window is found.

In [28]:
def check_inclusion(s1: str, s2: str) -> bool:
    if len(s1) > len(s2):
        return False

    need = [0] * 26
    window = [0] * 26

    for ch in s1:
        need[ord(ch) - 97] += 1

    k = len(s1)

    for right, ch in enumerate(s2):
        window[ord(ch) - 97] += 1

        if right >= k:
            window[ord(s2[right - k]) - 97] -= 1

        if right >= k - 1 and window == need:
            return True

    return False

print(check_inclusion("ab", "eidbaooo"))  # True
print(check_inclusion("ab", "eidboaoo"))  # False

True
False


## 12. Minimum window substring — advanced

Find the smallest substring of `s` that contains every character of `t`, including duplicates.

Maintain:
- `need`: required character frequencies.
- `window`: current window frequencies.
- `formed`: how many distinct required characters currently meet their required frequency.
- `required`: number of distinct characters in `need`.

Expand right until all requirements are met. Then shrink left while the window remains valid, recording the smallest answer.

In [29]:
from collections import Counter

def min_window(s: str, t: str) -> str:
    if not s or not t or len(t) > len(s):
        return ""

    need = Counter(t)
    window = {}
    required = len(need)
    formed = 0

    left = 0
    best_len = float("inf")
    best_start = 0

    for right, ch in enumerate(s):
        window[ch] = window.get(ch, 0) + 1

        if ch in need and window[ch] == need[ch]:
            formed += 1

        while formed == required:
            current_len = right - left + 1

            if current_len < best_len:
                best_len = current_len
                best_start = left

            outgoing = s[left]
            window[outgoing] -= 1

            if outgoing in need and window[outgoing] < need[outgoing]:
                formed -= 1

            left += 1

    if best_len == float("inf"):
        return ""

    return s[best_start:best_start + best_len]

print(min_window("ADOBECODEBANC", "ABC"))  # BANC
print(min_window("a", "aa"))              # 

BANC



## 13. Longest repeating-character replacement

Given an uppercase string and an integer `k`, find the longest substring that can be made of one repeated character by replacing at most `k` characters.

For a window:
- `window_len = right - left + 1`
- `max_freq` = highest frequency of any character in the window
- Replacements needed = `window_len - max_freq`

The window is valid when `window_len - max_freq <= k`.

The maximum frequency can be kept as a historical maximum in this algorithm; it need not be recomputed on every shrink.

In [30]:
def character_replacement(s: str, k: int) -> int:
    freq = {}
    left = 0
    max_freq = 0
    best = 0

    for right, ch in enumerate(s):
        freq[ch] = freq.get(ch, 0) + 1
        max_freq = max(max_freq, freq[ch])

        while (right - left + 1) - max_freq > k:
            outgoing = s[left]
            freq[outgoing] -= 1
            left += 1

        best = max(best, right - left + 1)

    return best

print(character_replacement("ABAB", 2))   # 4
print(character_replacement("AABABBA", 1)) # 4

4
4


## 14. When sliding window is not the right tool

Be careful:
- Exact subarray sum with negative values → prefix sum + hash map.
- Shortest subarray with sum at least `k` when negatives are allowed → prefix sums + monotonic deque (advanced).
- Maximum sum of any subarray → Kadane's algorithm.
- A fixed-size window can still work with negative values because its boundaries are predetermined.
- A variable-size sum window needs a justified monotonic property.

A hash map does not automatically make every window algorithm correct. The boundary movement rule must be valid.

## 15. Complexity

For the standard examples:
- Longest substring without repeats: expected `O(n)` time, `O(min(n, alphabet))` map space.
- At most K distinct: expected `O(n)` time, `O(K)` map space.
- Count at most K distinct: expected `O(n)` time, `O(K)` map space.
- Find all anagrams using 26-count comparisons: `O(26n) = O(n)` for fixed alphabet, `O(26)` space.
- Minimum window substring: expected `O(|s| + |t|)` time, space proportional to distinct characters.

Each element enters and leaves a normal sliding window at most once. Python dictionary operations are expected `O(1)` on average.

## 16. Common mistakes

1. Moving `left` when a previous duplicate is already outside the current window.
2. Forgetting to delete a key when its frequency becomes zero.
3. Using `if` where a `while` is needed to restore window validity.
4. Comparing only distinct characters when multiplicity matters.
5. Forgetting that anagram windows must have exactly the pattern length.
6. Confusing “at most K” with “exactly K”.
7. Using a variable-size sum window with negative numbers without proof.
8. Updating the best answer before the window satisfies the required condition.

## 17. Interview recognition checklist

Look for:
- “longest substring with ...”
- “smallest window containing ...”
- “at most K distinct”
- “exactly K distinct”
- “permutation” or “anagram substring”
- “replace at most K characters”
- “contiguous range of fixed length”

Ask:
1. Is the window fixed or variable size?
2. What information must the window maintain?
3. What makes a window valid?
4. When can the left boundary safely move?
5. Do duplicates and frequencies matter?
6. Is the alphabet restricted?

**Core mental model:** Expand right, update state, shrink left while invalid (when the problem's assumptions justify it), then update the answer.

## 18. Practice questions — recommended order

### Easy
1. [643. Maximum Average Subarray I](https://leetcode.com/problems/maximum-average-subarray-i/)
2. [219. Contains Duplicate II](https://leetcode.com/problems/contains-duplicate-ii/)

### Medium
3. [3. Longest Substring Without Repeating Characters](https://leetcode.com/problems/longest-substring-without-repeating-characters/)
4. [567. Permutation in String](https://leetcode.com/problems/permutation-in-string/)
5. [438. Find All Anagrams in a String](https://leetcode.com/problems/find-all-anagrams-in-a-string/)
6. [424. Longest Repeating Character Replacement](https://leetcode.com/problems/longest-repeating-character-replacement/)
7. [340. Longest Substring with At Most K Distinct Characters](https://leetcode.com/problems/longest-substring-with-at-most-k-distinct-characters/) — Premium
8. [992. Subarrays with K Different Integers](https://leetcode.com/problems/subarrays-with-k-different-integers/)
9. [76. Minimum Window Substring](https://leetcode.com/problems/minimum-window-substring/)

Suggested order: 643 → 219 → 3 → 567 → 438 → 424 → 992 → 76.

For every question, identify the window state and prove why moving the left boundary is safe.

## 19. Reusable templates

### Variable window with a frequency map
```python
def longest_at_most_k_distinct(s: str, k: int) -> int:
    freq = {}
    left = best = 0

    for right, ch in enumerate(s):
        freq[ch] = freq.get(ch, 0) + 1

        while len(freq) > k:
            outgoing = s[left]
            freq[outgoing] -= 1
            if freq[outgoing] == 0:
                del freq[outgoing]
            left += 1

        best = max(best, right - left + 1)

    return best
```

### Exactly K using at-most
```python
def exactly_k(s, k):
    return at_most_k(s, k) - at_most_k(s, k - 1)
```

### Fixed window update
```python
# Add nums[right].
# If the window is too large, remove nums[right - k].
# Evaluate once the window has exactly k elements.
```

You now have the core sliding-window + hashing patterns. Practice the questions before moving to the next concept.